In [ ]:
import pandas as pd
import os
import random

import numpy as np
from sklearn.preprocessing import StandardScaler
import geopandas as gpd

from spreg import GM_Combo, gets_sdm, GM_Lag
import libpysal
from spreg import OLS
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import rbf_kernel
from esda.moran import Moran

In [ ]:
# ============================================================
# 1. Maximum Mean Discrepancy (MMD)
# ============================================================

def mmd_rbf(X, Y, gamma=1.0):
    Kxx = rbf_kernel(X, X, gamma=gamma)
    Kyy = rbf_kernel(Y, Y, gamma=gamma)
    Kxy = rbf_kernel(X, Y, gamma=gamma)
    return Kxx.mean() + Kyy.mean() - 2 * Kxy.mean()


# ============================================================
# 2. CORAL Distance (Correlation Alignment)
# ============================================================

def coral_distance(X, Y):
    # means
    mean_diff = np.linalg.norm(X.mean(0) - Y.mean(0))

    # covariance differences
    cov_x = np.cov(X, rowvar=False)
    cov_y = np.cov(Y, rowvar=False)

    cov_diff = np.linalg.norm(cov_x - cov_y, ord='fro')

    return mean_diff + cov_diff


# ============================================================
# 3. Fréchet Distance (FID)
# ============================================================

from scipy.linalg import sqrtm

def frechet_distance(X, Y):
    eps = 1e-6

    X = (X - X.mean(0)) / (X.std(0) + 1e-6)
    Y = (Y - Y.mean(0)) / (Y.std(0) + 1e-6)

    mu_x, mu_y = X.mean(axis=0), Y.mean(axis=0)
    cov_x, cov_y = np.cov(X, rowvar=False) + eps * np.eye(X.shape[1]), np.cov(Y, rowvar=False) + eps * np.eye(Y.shape[1])

    # sqrt of covariance product
    cov_xy = sqrtm(cov_x.dot(cov_y))

    # handle numerical issues (imag part)
    if np.iscomplexobj(cov_xy):
        cov_xy = cov_xy.real

    fid = np.sum((mu_x - mu_y) ** 2) + \
          np.trace(cov_x + cov_y - 2 * cov_xy)
    return fid


# ============================================================
# 4. CKA Similarity (Centered Kernel Alignment)
# ============================================================

def cka_linear(X, Y):
    # centered Gram matrices
    X_centered = X - X.mean(axis=0)
    Y_centered = Y - Y.mean(axis=0)

    K = X_centered @ X_centered.T
    L = Y_centered @ Y_centered.T

    # Hilbert-Schmidt independence criterion
    numerator = np.sum(K * L)
    denominator = np.sqrt(np.sum(K * K) * np.sum(L * L))

    return numerator / denominator

In [ ]:
def load_state_areas(if_shuffle=True, state_codes:list = None):
    # areas = os.listdir("data")
    areas = [area for area in os.listdir("../../../data") if area.startswith(tuple(state_codes))]
    if if_shuffle:
        random.shuffle(areas)
    return areas

In [ ]:
def load_area_features(areas):
    demos_list = []
    pois_list = []
    dis_list = []
    for area in areas:
        if area == ".DS_Store":
            continue
        demos = np.load(f"../../../data/{area}/demos.npy")
        pois = np.load(f"../../../data/{area}/pois.npy")
        # dis = np.load(f"../../../data/{area}/dis.npy")

        demos_list.append(demos)
        pois_list.append(pois)
        # dis_list.append(dis)

    demos = np.concatenate(demos_list, axis=0)
    pois = np.concatenate(pois_list, axis=0)
    # dis = np.concatenate(dis_list)

    return demos, pois

In [ ]:
train_states = ['04']

In [ ]:
train_areas = load_state_areas(if_shuffle=False, state_codes = train_states)

In [ ]:
src_demos, src_pois = load_area_features(train_areas)

In [ ]:
df_us_states = pd.read_csv("../data/us_states_fips.csv", dtype={"FIPS": str})
test_states = list(set(df_us_states['FIPS']) - set(train_states))
test_areas = load_state_areas(if_shuffle=False, state_codes = test_states)

In [ ]:
def src_tar_diff(src_demos, src_pois, test_areas):
    rows = []
    for area in test_areas:
        tar_demos, tar_pois = load_area_features([area])

        demo_mmd = mmd_rbf(src_demos, tar_demos)
        poi_mmd = mmd_rbf(src_pois, tar_pois)

        demo_cd = coral_distance(src_demos, tar_demos)
        poi_cd = coral_distance(src_pois, tar_pois)

        demo_fd = frechet_distance(src_demos, tar_demos)
        poi_fd = frechet_distance(src_pois, tar_pois)

        row = {'GEOID': area,
               'demo_mmd': demo_mmd,
               'poi_mmd': poi_mmd,
               'demo_coral_distance': demo_cd,
               'poi_coral_distance': poi_cd,
               'demo_frechet_distance': demo_fd,
               'poi_frechet_distance': poi_fd,
              }
        rows.append(row)

    df_src_tar_diff = pd.DataFrame(rows)
    return df_src_tar_diff

In [ ]:
df_src_tar_diff = src_tar_diff(src_demos, src_pois, test_areas)

In [ ]:
df_src_tar_diff.head()

,GEOID,demo_mmd,poi_mmd,demo_coral_distance,poi_coral_distance,demo_frechet_distance,poi_frechet_distance
0,48123,0.200798,0.201143,5.774776e+07,9612.593470,2.692712e+07,9699.237718
1,13089,0.007630,0.008603,1.270624e+08,9267.838027,1.249269e+07,6697.816991
2,20099,0.125798,0.126714,1.134060e+08,9623.804317,3.832625e+07,9558.934813
3,18005,0.067465,0.067756,7.387944e+07,9660.436375,2.643647e+07,10586.499264
4,19123,0.143655,0.144001,1.795497e+08,9669.000173,9.750391e+07,10891.908514


In [ ]:
us_county_geo_file = '../geo_data/tl_2018_us_county/tl_2018_us_county.shp'
gpd_us_county =  gpd.read_file(us_county_geo_file)
gpd_us_county  = gpd_us_county [~gpd_us_county['STATEFP'].isin(['02','15','60','66','69','72','78'])]

eval_results = pd.read_csv('tables/source_{}_county_level.csv'.format('+'.join(train_states)), dtype={"GEOID": str})
eval_results = eval_results[~eval_results['GEOID'].str.startswith(('02','15','60','66','69','72','78'))]
avg_eval = eval_results.drop(columns = ['GEOID', 'num_regions'])

gpd_eval_results = pd.merge(eval_results, gpd_us_county[['STATEFP', 'COUNTYFP', 'GEOID', 'NAME', 'geometry']], on="GEOID", how="left")
metrics = ['CPC', 'RMSE', 'NRMSE', 'JSD_inflow', 'JSD_outflow', 'JSD_ODflow']
gpd_eval_results_vis = gpd_eval_results[['STATEFP', 'COUNTYFP', 'GEOID', 'NAME', 'geometry'] + metrics]
# gpd_eval_results_vis = pd.concat([gpd_eval_results_vis, pd.DataFrame(state_rows)], ignore_index=True)
# gpd_eval_results_vis[metrics] = gpd_eval_results_vis[metrics].fillna(-1)
gpd_eval_results_vis = gpd.GeoDataFrame(gpd_eval_results_vis, geometry="geometry")

In [ ]:
feat_scaler = StandardScaler()

df_analysis = pd.merge(gpd_eval_results_vis[gpd_eval_results_vis['CPC'] != -1], df_src_tar_diff, left_on="GEOID", right_on="GEOID", how="left")
# 'demo_mmd', 'demo_frechet_distance',
predictors = ['poi_mmd',
              'demo_coral_distance',
              'poi_coral_distance',
              'poi_frechet_distance']
# predictors =  ['poi_mean_diff_{}'.format(i) for i in range(src_pois.shape[1])]
df_analysis_per = df_analysis[metrics + predictors + ['geometry']]
df_analysis_per[predictors] = feat_scaler.fit_transform(df_analysis_per[predictors] )

/Users/zzhiyong/opt/anaconda3/envs/ox/lib/python3.13/site-packages/geopandas/geodataframe.py:1968: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


In [ ]:
gdf_analysis_per = gpd.GeoDataFrame(df_analysis_per, geometry="geometry")
gdf_analysis_per = gdf_analysis_per[~gdf_analysis_per.geometry.isna()]
gdf_analysis_per = gdf_analysis_per.to_crs(epsg=5070)
gdf_analysis_per = gdf_analysis_per[~gdf_analysis_per.geometry.isna()]
coords = np.array([gdf_analysis_per.centroid.x, gdf_analysis_per.centroid.y]).T

In [ ]:
W = libpysal.weights.KNN.from_array(coords, k=8)
W.transform = "r"

X = gdf_analysis_per[predictors]

y = gdf_analysis_per['RMSE'].apply(np.log1p).values.reshape((-1,1))

# model = GM_Combo_Het(y, X, w=W)
model = gets_sdm(y, X, w=W)[1]

ols = OLS(y, X)

Model selected by GETS-SDM: SARSAR
REGRESSION RESULTS
------------------

SUMMARY OF OUTPUT: SPATIALLY WEIGHTED 2SLS- GM-COMBO MODEL (HET)
----------------------------------------------------------------
Data set            :     unknown
Weights matrix      :       False
Dependent Variable  :     dep_var                Number of Observations:        2239
Mean dependent var  :      4.1718                Number of Variables   :           6
S.D. dependent var  :      0.6678                Degrees of Freedom    :        2233
Pseudo R-squared    :      0.5521
Spatial Pseudo R-squared:  0.4560
N. of iterations    :           1                Step1c computed       :          No

------------------------------------------------------------------------------------
            Variable     Coefficient       Std.Error     z-Statistic     Probability
------------------------------------------------------------------------------------
            CONSTANT         1.47094         0.11554        12.7

In [ ]:
# Compute VIF for each variable
from statsmodels.stats.outliers_influence import variance_inflation_factor
vif_df = pd.DataFrame()
vif_df["Variable"] = X.columns
vif_df["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

In [ ]:
vif_df

,Variable,VIF
0,poi_mmd,1.013868
1,demo_coral_distance,1.012506
2,poi_coral_distance,19.972082
3,poi_frechet_distance,19.971598


In [ ]:
rho = model.betas[-1]
coef = model.output.iloc[:-1, :].copy()
coef['total_impact'] = coef.loc[:, 'coefficients']/ rho
coef['source'] = ['+'.join(train_states)] * coef.shape[0]
# coef_dfs.append(coef)

# lag_pr2s.append(model.pr2)
# lag_pr2_es.append(model.pr2_e)
# ols_r2s.append(ols.r2)

mi_lag = Moran(model.u, W)
# lag_morans.append(mi_lag.I)
# lag_moran_pvals.append(mi_lag.p_norm)

mi_ols = Moran(ols.u, W)
# ols_morans.append(mi_ols.I)
# ols_moran_pvals.append(mi_ols.p_norm)

In [ ]:
print("Spatial Model Moran I:", mi_lag.I, "p-value:", mi_lag.p_norm)
print("OLS Moran I:", mi_ols .I, "p-value:", mi_ols.p_norm)

Spatial Model Moran I: 0.0008547946384839893 p-value: 0.8981177929451041
OLS Moran I: 0.21613569989636994 p-value: 1.0196616877192732e-100
